In [ ]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *
from textblob import TextBlob

#pd.set_option("display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/code/Initialization/Sentiment/Functions.ipynb"
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)
df_companies=pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SPXCompaniesByMarketCap.pkl")

In [ ]:
df_companies.reset_index(inplace=True)#.loc[:"2020-06-01"].tail(50)#.groupby("date").sum().plot()

In [ ]:
df_companies.reset_index().loc[:"2020-06-01"].groupby("symbol").mean().reset_index().plot(x="marketCap", y="symbol", kind="scatter",figsize=(10,30))#.groupby("symbol").plot()

In [ ]:
mean_prc = df_articles['PressReleaseCount'].mean()
# New words and values
new_words = {
    'crushes': 100,
    'beats': 5,
    'misses': -5,
    'trouble': -10,
    'falls': -100,
    "dividends":100,
    "dividend":100,
    "earning":100,
    "ipo":100,
    "alliance":100,
    "enhance":100,
    "enhanc":100
}
# Instantiate the sentiment intensity analyzer with the existing lexicon
vader = SentimentIntensityAnalyzer()
# Update the lexicon
vader.lexicon.update(new_words)

In [ ]:
# df_companies.groupby("date").sum().loc[2]#loc[df_companies.index.get_level_values("date")==date].values.sum()

In [ ]:
def marketCapFactor(companies,date):
    marketcap = 0
    for company in companies:
        try:
            marketcap += df_companies.loc[company, date].values[0]
        except:
            print("Exception")
            print(company)
            print(date)
    totalMarketCap = df_companies.loc[df_companies.index.get_level_values("date")==date].values.sum()
    print(totalMarketCap)
    return (marketcap/totalMarketCap)

# 1) DATE

In [ ]:
pos_word_list=[]
neu_word_list=[]
neg_word_list=[]
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['Sentiment']] = ''
df[['Adjectives']] = ''
df[['Verbs']] = ''
df[['Adverbs']] = ''
df[['Nouns']] = ''
df[["Articles"]] = ''
df[["polarity"]] = ''
df[["subjectivity"]] = ''
df[["polarityB"]] = ''

In [ ]:
for date, row in df_articles.T.iteritems():
    # try:
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    prc = row['PressReleaseCount']
    sentence = str(preprocessing_text(sentence))
    print("#0")
    sentence = sentence.lower()
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date,"Articles"] = sentence
    ss = vader.polarity_scores(sentence)
    print("#1")
    # for word in sentence.split():
    #     # print(word)
    #     if (vader.polarity_scores(word)['compound']) >= 0.5:
    #         pos_word_list.append(word)
    #     elif (vader.polarity_scores(word)['compound']) <= -0.5:
    #         neg_word_list.append(word)
    #     else:
    #         neu_word_list.append(word)  
    # pd.unique(pos_word_list)
    # pd.unique(neu_word_list)
    # pd.unique(neg_word_list)

    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = marketCapFactor(companieslist, date)
    print("#2")
    df.at[date, 'MarketCapFactor'] = mcf
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'Sentiment'] = (((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc)))*mcf
    if(len(adjectives(str(sentence)))):
        df.at[date, 'Adjectives'] = pd.unique(str(adjectives(str(sentence))).strip("[]"))
    if(len(verbs(str(sentence)))):
        df.at[date, 'Verbs'] = pd.unique(str(verbs(str(sentence))).strip("[]"))
    if(len(adverbs(str(sentence)))):
        df.at[date, 'Adverbs'] = pd.unique(str(adverbs(str(sentence))).strip("[]"))
    if(len(nouns(str(sentence)))):
        df.at[date, 'Nouns'] =  pd.unique(str(nouns(str(sentence))).strip("[]"))

    print("#3")
    #print(sentence)
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print("#4")
    df.at[date,"polarity"] = blob.sentiment.polarity
    print("#5")
    df.at[date,"subjectivity"] = blob.sentiment.subjectivity
    print("#6")
    if (blob.sentiment.polarity > 0 ):
        df.at[date,"polarityB"] = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        df.at[date,"polarityB"] = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        df.at[date,"polarityB"] = "Neutral"
    # except:
    #     print("Exception:"+date)
df['Sentiment'].apply(np.float)
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment.pkl")

In [ ]:
# del df
# del df_articles

In [ ]:
df_companies

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
df["Negative"].mean()

# 2) HOUR

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDateHourDF.pkl', compression=None)
del df 

## Vader

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

rows_list = []
for date, row in df_articles.T.iteritems():
    Hour = int(row["Hour"])
    print(Hour)
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")

    prc = row['PressReleaseCount']
    ss = vader.polarity_scores(sentence)

    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = marketCapFactor(companieslist, date)
    MarketCapFactor = mcf
    Sentiment=0
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            Sentiment = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))*mcf

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    print(date)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Date":date, "Hour": Hour, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "sentiment":Sentiment, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"Articles":sentence, "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "PressReleaseCount":row["PressReleaseCount"], "MarketCapFactor":MarketCapFactor}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","Articles","Title","Text", "Company", "PressReleaseCount", "MarketCapFactor"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourly.pkl")

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

In [ ]:
del df

## General Inquirer 

In [ ]:
import pysentiment2 as ps
hiv4 = ps.HIV4()
rows_list = []
for date, row in df_articles.T.iteritems():
    Hour = int(row["Hour"])
    print(Hour)
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")

    prc = row['PressReleaseCount']
    tokens = hiv4.tokenize(str(sentence))  # text can be tokenized by other ways
                                # however, dict in HIV4 is preprocessed
                                # by the default tokenizer in the library
    score = hiv4.get_score(tokens)
    Neg = score['Negative']
    Pos = score['Positive']
    Polarity = score['Polarity']
    Subjectivity = score['Subjectivity']
    companieslist = list(filter(None, row['Company'].split(". ")))
    mcf = marketCapFactor(companieslist, date)
    MarketCapFactor = mcf
    Sentiment=0
    if ((Pos + Neg) != 0):
        number = 100*(Pos / (Pos + Neg))
        if (number != ''):
            Sentiment = ((100*(Neg / (Pos + Neg)))*(prc/mean_prc))*mcf

    new_row = {"Date":date, "Hour": Hour,"neg":Neg, "pos":Pos, "sentiment":Sentiment, "polarity": Polarity, "subjectivity":Subjectivity, "Articles":sentence, "Title":row["Title"], "Text":row["Text"], "Company":row["Company"], "PressReleaseCount":row["PressReleaseCount"], "MarketCapFactor":MarketCapFactor}
    rows_list.append(new_row)
df = pd.DataFrame(data=rows_list, columns=["Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","Articles","Title","Text", "Company", "PressReleaseCount", "MarketCapFactor"])
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentHourlyGeneralInquirer.pkl")

In [ ]:
df["positive"].()

# 3) TRADING HOURS 09-16

In [ ]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_09_15.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'Articles'] = sentence
    prc = row['PressReleaseCount']

    ss = vader.polarity_scores(sentence)

    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_09_15.pkl")

In [ ]:
del df

# 4) Not Trading Hours 16-09

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate_16_08.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))
    sentence = sentence.strip("[]")
    sentence = sentence.strip("''")
    df.at[date, 'Articles'] = sentence
    prc = row['PressReleaseCount']

    ss = vader.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = ((100*(ss['neg'] / (ss['pos'] + ss['neg'])))*(prc/mean_prc))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']

In [ ]:
df_articles.drop(columns={"Articles"},inplace=True)
df = df.join(df_articles)

In [ ]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/Sentiment_16_08.pkl")

In [ ]:
del df

# 5) TIME

In [ ]:
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/df_dates.pkl', compression=None)

In [ ]:
df_articles

In [ ]:
# del df

# df  = pd.DataFrame()
# # Adding new columns to the data frame
# df[['compound']] = ''
# df[['neg']] = ''
# df[['neu']] = ''
# df[['pos']] = ''
# df[['sentiment']] = ''

In [ ]:
df_articles.loc[df_articles["Date"]>"2020-06-01"]

In [ ]:
debug=False
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

rows_list = []
for date, row in df_articles.T.iteritems():

    sentence = unicodedata.normalize('NFKD', row['Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = str(preprocessing_text(sentence))

    ss = vader.polarity_scores(sentence)

    Hour = row["Hour"]
    Time = row["Time"]
    Date = row["Date"]
    Compound = ss['compound']
    Neg = ss['neg']
    Neu = ss['neu']
    Pos = ss['pos']
    Sentiment=0
    if ((Pos + Neg) != 0):
        number = 100*(Pos / (Pos + Neg))
        if (number != ''):
            SentimentScore = 100*((Neg + Neu) / (Pos + Neg + Neu))
    two_grams = generate_ngrams(sentence,2)

    # countNeg+=ss['neg']
    # countNeu+=ss['neu']
    # countPos+=ss['pos']
    
    ############## TEXTBLOB ##############
    blob = TextBlob(sentence)
    Polarity = blob.sentiment.polarity
    Subjectivity = blob.sentiment.subjectivity
    polarityB = ""
    if (blob.sentiment.polarity > 0 ):
        polarityB = "Positive"
    elif (blob.sentiment.polarity < 0 ):
        polarityB = "Negative"
    elif (blob.sentiment.polarity == 0 ):
        polarityB = "Neutral"

    new_row = {"Time":Time, "Date":Date, "Hour": Hour, "Sentiment":Sentiment, "compound":Compound, "neg":Neg, "neu":Neu, "pos":Pos, "SentimentScore":SentimentScore, "polarity": Polarity, "subjectivity":Subjectivity, "polarityB":polarityB,"Articles":row["Articles"], "Title":row["Title"], "Text":row["Text"], "Company":row["Company"]}
    rows_list.append(new_row)

In [ ]:
df = pd.DataFrame(data=rows_list, columns=["Time", "Date", "Hour", "compound", "neg", "neu", "pos", "sentiment", "polarity", "subjectivity", "polarityB","Articles","Title","Text", "Company"])

In [ ]:
# print("Number of Negative words="+str(countNeg))
# print("Number of Positive words="+str(countPos))
# print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")

In [ ]:
df["sentiment"].ewm(10).mean().plot(figsize=(20,10))

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/SentimentTime.pkl")

In [ ]:
df.index = df.Time

In [ ]:
df.loc[df["Company"]=="BBY"]["polarity"].plot(figsize=(20,10))

In [ ]:
df["sentiment"].ffill(inplace=True)